# Amyloid pLM Pipeline — Kaggle (2× T4)

Full pipeline using **both T4 GPUs**:
- **Stage 1 (training embeddings):** the short training peptides are embedded once with
  ProtT5 on **GPU 0** and ESM2 on **GPU 1** in parallel, and cached.
- **Stages 2–3 (TensorFlow):** grid search → ensemble (5-fold CV).
- **Stage 4 (benchmark):** the PLMs run inference **directly on each 18-residue sliding
  window** (ProtT5 on GPU 0, ESM2 on GPU 1) — no cache; windows are embedded live and scored.

### Before running
1. **Settings → Accelerator → `GPU T4 x2`**
2. **Settings → Internet → On**
3. **Add Input → your Dataset** with the `New_dataset/` JSON files

Outputs: `/kaggle/working/results/<run_id>/{training,benchmark}/`.

## 0 · GPU check

In [ ]:
!nvidia-smi -L
import torch
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| visible CUDA GPUs:", N_GPU)
assert N_GPU >= 1, "No GPU. Settings -> Accelerator -> GPU T4 x2."
if N_GPU < 2:
    print("WARNING: only 1 GPU visible; pick 'GPU T4 x2' to use both.")

## 1 · Clone repo & install deps

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/NGUYENTHAISON-2311/dl_plm_pipeline.git"
REPO_DIR = "/kaggle/working/dl_plm_pipeline"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
# Args passed as a list (no shell) -> NO quotes; the version spec has no spaces.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.30", "sentencepiece", "h5py", "pyyaml"], check=False)
print("repo ready:", REPO_DIR)


## 2 · Configure paths (Kaggle dataset in, /kaggle/working out)

In [ ]:
import glob, os, shutil
from src.utils.config import load_config, resolve_path
from src.utils.seed import set_global_seed

# ============================================================================
# EDIT ME: where the training-peptide embedding cache lives.
#   - leave as-is to generate/store under /kaggle/working, OR
#   - point straight at an attached dataset file, e.g.
#     "/kaggle/input/my-embeddings/embeddings.h5"
EMBEDDINGS_CACHE = "/kaggle/working/embeddings.h5"
# ============================================================================

cfg = load_config()
set_global_seed(cfg.get("seed", 42))

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found under /kaggle/input — attach New_dataset via 'Add Input'.")
    return hits[0]

cfg["paths"]["train_pos"]                = find("train_core_set_KA.json")
cfg["paths"]["train_neg"]                = find("disordered_regions_no_core_overlaps_filtered_clustered.json")
cfg["paths"]["classification_benchmark"] = find("new_representative_classification_benchmark_KA.json")
cfg["paths"]["position_benchmark"]       = find("new_representative_positive_benchmark_KA.json")

# Auto-pick up a precomputed embeddings.h5 if you attached one and left the default path.
if EMBEDDINGS_CACHE == "/kaggle/working/embeddings.h5":
    pre = glob.glob("/kaggle/input/**/embeddings.h5", recursive=True)
    if pre and not os.path.exists(EMBEDDINGS_CACHE):
        shutil.copy(pre[0], EMBEDDINGS_CACHE)
        print("copied precomputed embeddings:", pre[0])
    elif not pre:
        print("no precomputed embeddings.h5 under /kaggle/input -> Stage 1 will generate them")

cfg["embeddings"]["cache"]  = EMBEDDINGS_CACHE
cfg["paths"]["results_dir"] = "/kaggle/working/results"
for k, v in cfg["paths"].items():
    print(f"{k:28s}: {v}")
print("embeddings cache:", cfg["embeddings"]["cache"], "| exists:", os.path.exists(EMBEDDINGS_CACHE))


## 3 · Run configuration

In [ ]:
import datetime
from pathlib import Path

RUN_GRIDSEARCH = True
LIMIT          = None
MODELS         = cfg["gridsearch"]["models"]
EPOCHS         = cfg["train"]["epochs"]
USE_MIRRORED   = False

cfg["gridsearch"]["models"] = MODELS
cfg["train"]["epochs"]      = EPOCHS
cfg["train"]["distribute"]  = "mirrored" if USE_MIRRORED else "none"

RUN_ID    = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
run_dir   = Path(cfg["paths"]["results_dir"]) / RUN_ID
train_dir = run_dir / "training"
bench_dir = run_dir / "benchmark"
for d in (train_dir / "plots", bench_dir / "plots"):
    d.mkdir(parents=True, exist_ok=True)
cfg["paths"]["results_dir"] = str(train_dir)

print("RUN_ID :", RUN_ID, "| models:", MODELS, "| epochs:", EPOCHS, "| gridsearch:", RUN_GRIDSEARCH)
print("out    :", run_dir)

## 4 · Stage 1 — embed TRAINING peptides on **both T4 GPUs** (PyTorch)

ProtT5 → `cuda:0`, ESM2 → `cuda:1`, in parallel. Cached to `/kaggle/working/embeddings.h5`.
The benchmark is NOT embedded here — its windows are embedded live in Stage 4.

In [ ]:
from src.data.dataset import load_training_peptides, load_benchmark, assert_no_leakage
from src.data.embeddings import EmbeddingCache, generate_embeddings, generate_embeddings_dual_gpu

peptides  = load_training_peptides(resolve_path(cfg, "train_pos"), resolve_path(cfg, "train_neg"))
benchmark = load_benchmark(resolve_path(cfg, "classification_benchmark"))
peptides  = assert_no_leakage(peptides, benchmark, drop=True)
if LIMIT:
    peptides = peptides[:LIMIT]
print(f"{len(peptides)} training peptides | {len(benchmark)} benchmark sequences")

cache    = EmbeddingCache(cfg["embeddings"]["cache"])
all_seqs = [p.sequence for p in peptides]      # training peptides only (benchmark is live)
missing  = cache.missing(all_seqs)
print(f"cache: {cache.path} | {len(missing)} of {len(all_seqs)} peptides missing")

if missing:
    # Not fully cached -> generate the missing peptide embeddings on the GPU(s).
    if N_GPU >= 2:
        generate_embeddings_dual_gpu(all_seqs, cfg, "cuda:0", "cuda:1")
    else:
        cfg["embeddings"]["device"] = "cuda" if torch.cuda.is_available() else "cpu"
        generate_embeddings(all_seqs, cfg)
else:
    print("all training-peptide embeddings already cached -> skipping PLM generation")

# Fail loudly here (not cryptically in Stage 2) if the cache is still incomplete.
still = cache.missing(all_seqs)
assert not still, (f"{len(still)} training peptides missing from {cache.path}. "
                   f"Attach data/processed/embeddings.h5 as a Kaggle Dataset, or let this "
                   f"cell generate them (needs GPU + Internet for the PLM weights).")
print("training-peptide cache complete:", cache.path)


Free PyTorch GPU memory before TensorFlow training (PLMs reload in Stage 4 for the live benchmark).

In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print("released torch CUDA memory")

## 5 · TensorFlow setup (memory growth so TF and torch can share the GPUs)

In [ ]:
import tensorflow as tf
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass
print("TF", tf.__version__, "| GPUs:", [d.name for d in tf.config.list_physical_devices("GPU")])

## 6 · Stage 2 — grid search (5-fold CV per architecture)

In [ ]:
import numpy as np
import pandas as pd
from src.training.grid_search import run_gridsearch, default_best_per_model
from src.utils.io import write_json

if RUN_GRIDSEARCH:
    best_per_model = run_gridsearch(cfg, cache, peptides)
else:
    best_per_model = default_best_per_model(cfg)
    write_json(best_per_model, train_dir / "best_per_model.json")
pd.DataFrame([{"model": m, **v["best_hp"]} for m, v in best_per_model.items()])

## 7 · Stage 3 — ensemble (best arch × 5 folds) + out-of-fold predictions

In [ ]:
from src.training.ensemble import build_ensemble_cv
ensemble, oof, per_model_fold = build_ensemble_cv(cfg, cache, peptides, best_per_model)
ensemble.save(train_dir / "ensemble")
print(f"{len(ensemble.members)} ensemble members ({len(best_per_model)} archs x {cfg['cv']['n_splits']} folds)")

## 8 · Training metrics — residue-level OOF (AUROC / PR / confusion)

In [ ]:
from src.evaluation.metrics import residue_metrics
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

thr = cfg["inference"]["residue_threshold"]
cv_df = pd.DataFrame(per_model_fold)
cv_summary = cv_df.groupby("model")[["auroc","auprc","mcc","f1","precision","recall"]].mean().round(3)
ens_metrics = residue_metrics(oof["labels"], oof["scores"], oof["mask"], threshold=thr)
print("Ensemble OOF residue metrics:")
for k, v in ens_metrics.items():
    print(f"  {k:10s}: {v}")

write_json({"ensemble_oof": ens_metrics, "per_model_fold": per_model_fold,
            "per_model_mean": cv_summary.reset_index().to_dict("records")}, train_dir / "metrics.json")
cv_df.to_csv(train_dir / "per_model_fold_metrics.csv", index=False)
np.savez_compressed(train_dir / "oof_residue_scores.npz", ids=np.array(oof["ids"]),
                    scores=oof["scores"], labels=oof["labels"], mask=oof["mask"])

m = oof["mask"].astype(bool)
yt = oof["labels"][m].ravel().astype(int); ys = oof["scores"][m].ravel(); yp = (ys > thr).astype(int)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(yt, ys, ax=ax[0]);        ax[0].set_title("ROC (residue OOF)")
PrecisionRecallDisplay.from_predictions(yt, ys, ax=ax[1]); ax[1].set_title("PR (residue OOF)")
ConfusionMatrixDisplay.from_predictions(yt, yp, ax=ax[2], colorbar=False, display_labels=["non-core","core"]); ax[2].set_title("Confusion (residue OOF)")
plt.tight_layout(); fig.savefig(train_dir / "plots" / "training_metrics.png", dpi=120); plt.show()
cv_summary

## 9 · Stage 4 — benchmark: **PLMs run live on each 18-residue window**

`PLMEmbedder` loads ProtT5 (GPU 0) + ESM2 (GPU 1) and embeds every sliding window directly;
the ensemble scores each window; overlapping scores are aggregated; the >0.5 / run>10 rule
gives the sequence call.

In [ ]:
from src.data.embeddings import PLMEmbedder
from src.evaluation.sliding import score_sequence_live
from src.evaluation.classify import classify_profile, sequence_score
from src.evaluation.metrics import sequence_metrics
from sklearn.metrics import roc_auc_score, average_precision_score

# ProtT5 on GPU 0, ESM2 on GPU 1 when two GPUs are present.
if N_GPU >= 2:
    embedder = PLMEmbedder(cfg, device_prott5="cuda:0", device_esm2="cuda:1")
else:
    embedder = PLMEmbedder(cfg, device="auto")

min_run = cfg["inference"]["min_consecutive"]
profiles, windows_out = {}, {}
y_true_seq, y_pred_seq, y_score_seq = [], [], []
for b in benchmark:
    profile, windows = score_sequence_live(b.sequence, embedder, ensemble, cfg)  # PLM live per window
    label, run, _ = classify_profile(profile, thr, min_run)
    sscore = sequence_score(profile, min_run)
    y_true_seq.append(b.label); y_pred_seq.append(label); y_score_seq.append(sscore)
    profiles[b.id]    = {"label_true": b.label, "label_pred": label, "longest_run": int(run),
                         "sequence_score": round(float(sscore), 4),
                         "profile": [round(float(x), 4) for x in profile]}
    windows_out[b.id] = windows

seq_m = sequence_metrics(y_true_seq, y_pred_seq)
seq_m["auroc"] = float(roc_auc_score(y_true_seq, y_score_seq)) if len(set(y_true_seq)) > 1 else float("nan")
seq_m["auprc"] = float(average_precision_score(y_true_seq, y_score_seq))
print("Benchmark sequence-level metrics:")
for k, v in seq_m.items():
    print(f"  {k:10s}: {v}")

write_json({"sequence_level": seq_m,
            "inference": {"residue_threshold": thr, "min_consecutive": min_run,
                          "window_len": cfg["window_len"], "stride": cfg["inference"]["window_stride"]},
            "ensemble_members": len(ensemble.members)}, bench_dir / "metrics.json")
write_json(profiles, bench_dir / "profiles.json")
write_json(windows_out, bench_dir / "windows.json")
pd.DataFrame({"id": [b.id for b in benchmark], "true": y_true_seq,
              "pred": y_pred_seq, "score": y_score_seq}).to_csv(bench_dir / "sequence_predictions.csv", index=False)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(y_true_seq, y_score_seq, ax=ax[0]);        ax[0].set_title("ROC (sequence)")
PrecisionRecallDisplay.from_predictions(y_true_seq, y_score_seq, ax=ax[1]); ax[1].set_title("PR (sequence)")
ConfusionMatrixDisplay.from_predictions(y_true_seq, y_pred_seq, ax=ax[2], colorbar=False, display_labels=["NONAMYLOID","AMYLOID"]); ax[2].set_title("Confusion (sequence)")
plt.tight_layout(); fig.savefig(bench_dir / "plots" / "benchmark_metrics.png", dpi=120); plt.show()

## 10 · Summary — saved artifacts

In [ ]:
print("RUN_ID:", RUN_ID, "\n")
for tag, d in (("TRAINING", train_dir), ("BENCHMARK", bench_dir)):
    print(f"{tag}: {d}")
    for p in sorted(Path(d).rglob("*")):
        if p.is_file():
            print("   ", p.relative_to(d))
    print()